In [2]:
import sqlite3
from sqlite3 import Error

 
def create_connection(db_file):
    """ create a database connection to the SQLite database
        specified by the db_file
    :param db_file: database file
    :return: Connection object or None
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        print(sqlite3.version)
    except Error as e:
        print(e)
 
    return conn

 
def select_all_tasks(conn):
    """
    Query all rows in the tasks table
    :param conn: the Connection object
    :return:
    """
    cur = conn.cursor()
    
    query1 = """
        SELECT *
        FROM FACILITIES
        """
    cur.execute(query1)
 
    rows = cur.fetchall()
 
    for row in rows:
        print(row)


def main():
    database = "sqlite_db_pythonsqlite.db"
 
    # create a database connection
    conn = create_connection(database)
    with conn: 
        print("2. Query all tasks")
        select_all_tasks(conn)
 
 
if __name__ == '__main__':
    main()

2.6.0
2. Query all tasks
(0, 'Tennis Court 1', 5, 25, 10000, 200)
(1, 'Tennis Court 2', 5, 25, 8000, 200)
(2, 'Badminton Court', 0, 15.5, 4000, 50)
(3, 'Table Tennis', 0, 5, 320, 10)
(4, 'Massage Room 1', 9.9, 80, 4000, 3000)
(5, 'Massage Room 2', 9.9, 80, 4000, 3000)
(6, 'Squash Court', 3.5, 17.5, 5000, 80)
(7, 'Snooker Table', 0, 5, 450, 15)
(8, 'Pool Table', 0, 5, 400, 15)


C:\TEmp\ipykernel_30216\1913378692.py:14: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(sqlite3.version)


In [6]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('sqlite_db_pythonsqlite.db')  # Replace with your actual database file

# Execute the SQL query
df = pd.read_sql_query("SELECT * FROM Facilities", conn)

# Display the results
df


,facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,0,Tennis Court 1,5.0,25.0,10000,200
1,1,Tennis Court 2,5.0,25.0,8000,200
2,2,Badminton Court,0.0,15.5,4000,50
3,3,Table Tennis,0.0,5.0,320,10
4,4,Massage Room 1,9.9,80.0,4000,3000
5,5,Massage Room 2,9.9,80.0,4000,3000
6,6,Squash Court,3.5,17.5,5000,80
7,7,Snooker Table,0.0,5.0,450,15
8,8,Pool Table,0.0,5.0,400,15


In [9]:
#Q10: Produce a list of facilities with a total revenue less than 1000.
#The output of facility name and total revenue, sorted by revenue. Remember
#that there's a different cost for guests and members! */
query = """
SELECT 
    f.name AS facility_name,
    SUM(
        CASE 
            WHEN b.memid = 0 THEN b.slots * f.guestcost
            ELSE b.slots * f.membercost
        END
    ) AS total_revenue
FROM Bookings b
JOIN Facilities f ON b.facid = f.facid
GROUP BY f.name
HAVING total_revenue < 1000
ORDER BY total_revenue;
"""
# Run the query and load into a DataFrame
df = pd.read_sql_query(query, conn)

# Display result
df

,facility_name,total_revenue
0,Table Tennis,180
1,Snooker Table,240
2,Pool Table,270


In [15]:
# connect to the database
conn = sqlite3.connect('sqlite_db_pythonsqlite.db')
# Q11: Produce a report of members and who recommended them in alphabetic surname,firstname order */

# First, let's check what tables are available in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
available_tables = cursor.fetchall()
print("Available tables:", [table[0] for table in available_tables])


# query
query = """
SELECT 
    CONCAT(m.firstname, ' ', m.surname) AS member_name,
    CONCAT(r.firstname, ' ', r.surname) AS recommended_by
FROM Members AS  m
LEFT JOIN Members r 
    ON m.recommendedby = r.memid
ORDER BY m.surname, m.firstname;
"""

# run and display
df = pd.read_sql_query(query, conn)
df

Available tables: ['Bookings', 'Facilities', 'Members']


,member_name,recommended_by
0,Florence Bader,Ponder Stibbons
1,Anne Baker,Ponder Stibbons
2,Timothy Baker,Jemima Farrell
3,Tim Boothe,Tim Rownam
4,Gerald Butters,Darren Smith
5,Joan Coplin,Timothy Baker
6,Erica Crumpet,Tracy Smith
7,Nancy Dare,Janice Joplette
8,David Farrell,
9,Jemima Farrell,


In [20]:

import pandas as pd
import sqlite3
#* Q12: Find the facilities with their usage by member, but not guests */

conn = sqlite3.connect('sqlite_db_pythonsqlite.db')  # Replace with your actual database file




query = """
SELECT 
     f.name AS facility_name,
     COUNT(b.bookid) AS member_usage_count
FROM Facilities f
JOIN Bookings b
    ON f.facid = b.facid
WHERE b.memid != 0
GROUP BY f.name
ORDER BY f.name
"""

# Execute the query and store results in a pandas DataFrame
df = pd.read_sql_query(query, conn)

# Display the results
print(df)

# Close the connection when done
conn.close()


     facility_name  member_usage_count
0  Badminton Court                 344
1   Massage Room 1                 421
2   Massage Room 2                  27
3       Pool Table                 783
4    Snooker Table                 421
5     Squash Court                 195
6     Table Tennis                 385
7   Tennis Court 1                 308
8   Tennis Court 2                 276


In [22]:
import sqlite3
import pandas as pd
#* Q13: Find the facilities usage by month, but not guests */
# Connect to your database
conn = sqlite3.connect('sqlite_db_pythonsqlite.db')  # Replace with your DB path

# SQL query
query = """
SELECT 
    f.name AS facility_name,
    strftime('%Y-%m', b.starttime) AS month,
    COUNT(b.bookid) AS usage_count
FROM Bookings b
JOIN Facilities f
    ON b.facid = f.facid
WHERE b.memid != 0
GROUP BY f.name, strftime('%Y-%m', b.starttime)
ORDER BY f.name, month;
"""

# Execute query and load into Pandas DataFrame
df = pd.read_sql_query(query, conn)

# Display the result
df

,facility_name,month,usage_count
0,Badminton Court,2012-07,51
1,Badminton Court,2012-08,132
2,Badminton Court,2012-09,161
3,Massage Room 1,2012-07,77
4,Massage Room 1,2012-08,153
5,Massage Room 1,2012-09,191
6,Massage Room 2,2012-07,4
7,Massage Room 2,2012-08,9
8,Massage Room 2,2012-09,14
9,Pool Table,2012-07,103
